# SkinScan AI — тренирање на модел за препознавање кожни болести

Овој notebook тренира **EfficientNet-B3** (истата архитектура како во leafscan) да препознава типови на кожни лезии од фотографија, комбинирајќи два датасета:

- **HAM10000** — ~10,000 дерматоскопски слики (7 категории)
- **PAD-UFES-20** — ~2,300 клинички слики снимени со обичен телефон (6 категории, важно затоа што одговара на тоа како луѓето реално ќе фотографираат со твојата апп)

## Како да го користиш ова (чекор по чекор)

1. Отвори [Google Colab](https://colab.research.google.com), избери **File → Upload notebook** и качи ја оваа `.ipynb` датотека.
2. Оди на **Runtime → Change runtime type** и избери **GPU (T4)** — бесплатно е.
3. Изврши ги ќелиите по ред одозгора надолу (Shift+Enter на секоја, или Runtime → Run all).
4. Кога ќе дојде до качување датотеки (Kaggle клуч, PAD-UFES-20 зип-ови), следи ги упатствата во таа ќелија.
5. На крајот, notebook-от ќе ти понуди да симнеш `skin_model.pt` и `label_converter.json` — тие два фајла ги ставаш во Django проектот (упатство подолу во последната ќелија).

Времетраење: околу 30–90 минути на бесплатен Colab GPU, во зависност од бројот на епохи.

## Важна напомена

Ова е модел за **прелиминарна проценка**, не медицинска дијагноза. Секоја апп изградена на него треба јасно да го каже тоа и секогаш да препорача преглед кај дерматолог, особено ако резултатот е меланом, базоцелуларен или планоцелуларен карцином.

## Чекор 0 — Инсталирање потребни библиотеки

In [ ]:
!pip install -q kaggle torch torchvision scikit-learn pandas pillow tqdm
print("Готово.")

## Чекор 1 — Симни го HAM10000 датасетот

Треба Kaggle API token (бесплатно):
1. Оди на https://www.kaggle.com/settings/api
2. Во полето за име впиши нешто (пр. `skinscan`) и кликни **Generate**
3. Ќе се прикаже token кој почнува со `KGAT_...` - копирај го
4. Изврши ја ќелијата подолу и залепи го токенот кога ќе побара


In [ ]:
import os

# Kaggle сега дава API token (стринг KGAT_...) наместо kaggle.json фајл за симнување.
# Ако сепак добиеш класичен kaggle.json фајл, гледај ја алтернативата во коментар подолу.

KAGGLE_TOKEN = input("Залепи го твојот Kaggle API token (KGAT_...): ").strip()

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/access_token", "w") as f:
    f.write(KAGGLE_TOKEN)
os.chmod("/root/.kaggle/access_token", 0o600)
print("Kaggle токенот е поставен.")

# Алтернатива ако добиеш класичен kaggle.json фајл наместо token:
# from google.colab import files
# uploaded = files.upload()
# with open("/root/.kaggle/kaggle.json", "wb") as f:
#     f.write(list(uploaded.values())[0])
# os.chmod("/root/.kaggle/kaggle.json", 0o600)


In [ ]:
!mkdir -p /content/data/ham10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data/ham10000 --unzip
print("HAM10000 симнат.")

## Чекор 2 — Симни го PAD-UFES-20 датасетот

Овој датасет се симнува рачно (Mendeley нема директен API):
1. Оди на https://data.mendeley.com/datasets/zr7vgbcyr2/1
2. Кликни **Download All** (ќе добиеш еден `.zip`, на пр. `zr7vgbcyr2-1.zip`)
3. Тој `.zip` качи го во твојот **Google Drive** (било која папка на MyDrive)
4. Во Google Drive, десен клик на фајлот → **Share** → под "General access" смени на **"Anyone with the link"** (Viewer) → **Copy link**
5. Изврши ја ќелијата подолу и залепи го копираниот линк кога ќе побара — ќе го симне директно преку линкот, без Drive авторизација (го заобиколуваме "credential propagation" проблемот со `drive.mount()`).

In [ ]:
import os, zipfile, glob, re

os.makedirs("/content/data/padufes", exist_ok=True)

!pip -q install gdown
import gdown

print("Во Google Drive: десен клик на zip-от -> Share -> 'Anyone with the link' -> Copy link")
drive_link = input("Залепи го Google Drive линкот тука: ").strip()

m = re.search(r"/d/([a-zA-Z0-9_-]+)", drive_link) or re.search(r"id=([a-zA-Z0-9_-]+)", drive_link)
assert m, "Не можам да го извлечам ID-то од линкот - провери дали е точно копиран целиот линк за споделување."
file_id = m.group(1)

dest_zip = "/content/data/padufes/padufes20_drive.zip"
gdown.download(id=file_id, output=dest_zip, quiet=False)
assert os.path.exists(dest_zip) and os.path.getsize(dest_zip) > 0, (
    "Симнувањето не успеа - провери дали пристапот е навистина 'Anyone with the link'."
)
print("Симнато:", dest_zip, os.path.getsize(dest_zip), "bytes")

# распакувај - повторувај додека не остане ниту еден нераспакуван zip
# (некои zip-ови содржат вгнездени zip-ови, на пр. imgs_part_1.zip внатре во главниот zip)
while True:
    zips = glob.glob("/content/data/padufes/**/*.zip", recursive=True)
    if not zips:
        break
    progressed = False
    for zpath in zips:
        try:
            with zipfile.ZipFile(zpath, "r") as z:
                z.extractall(os.path.dirname(zpath))
            print(f"Распакувано: {zpath}")
            os.remove(zpath)
            progressed = True
        except Exception as e:
            print(f"Прескокнато (не е валиден zip): {zpath} -> {e}")
    if not progressed:
        break

print("PAD-UFES-20 подготвено.")

## Чекор 3 — Обединување на двата датасета

Двата датасета имаат различни имиња на категории, па ги мапираме кон заеднички список од 8 состојби. Кодот подолу автоматски пребарува низ сите преземени папки за да ги најде metadata CSV-фајловите и сликите, без разлика на точната внатрешна структура на папките (таа варира меѓу верзии на датасетите).

In [ ]:
import pandas as pd
import glob, os

# --- HAM10000 metadata ---
ham_csv_candidates = glob.glob("/content/data/ham10000/**/*metadata*.csv", recursive=True)
assert ham_csv_candidates, "Не е најдено HAM10000_metadata.csv - провери дали Чекор 1 помина успешно."
ham_meta = pd.read_csv(ham_csv_candidates[0])
print("HAM10000 metadata:", ham_meta.shape, "колони:", list(ham_meta.columns))

# индексирај ги сите слики по име на фајл (без екстензија) за брзо пребарување
ham_images = {}
for ext in ("*.jpg", "*.jpeg", "*.png"):
    for p in glob.glob(f"/content/data/ham10000/**/{ext}", recursive=True):
        ham_images[os.path.splitext(os.path.basename(p))[0]] = p
print("Најдени HAM10000 слики:", len(ham_images))

# --- PAD-UFES-20 metadata ---
pad_csv_candidates = [p for p in glob.glob("/content/data/padufes/**/*.csv", recursive=True)]
assert pad_csv_candidates, "Не е најдено metadata.csv за PAD-UFES-20 - провери дали Чекор 2 помина успешно."
pad_meta = pd.read_csv(pad_csv_candidates[0])
print("PAD-UFES-20 metadata:", pad_meta.shape, "колони:", list(pad_meta.columns))

pad_images = {}
for ext in ("*.png", "*.jpg", "*.jpeg"):
    for p in glob.glob(f"/content/data/padufes/**/{ext}", recursive=True):
        pad_images[os.path.splitext(os.path.basename(p))[0]] = p
print("Најдени PAD-UFES-20 слики:", len(pad_images))

In [ ]:
# --- мапирање кон заеднички категории ---

HAM10000_MAP = {
    "akiec": "actinic_keratosis",
    "bcc":   "basal_cell_carcinoma",
    "bkl":   "seborrheic_keratosis",
    "df":    "dermatofibroma",
    "mel":   "melanoma",
    "nv":    "nevus",
    "vasc":  "vascular_lesion",
}

PAD_UFES_MAP = {
    "ACK": "actinic_keratosis",
    "BCC": "basal_cell_carcinoma",
    "SCC": "squamous_cell_carcinoma",
    "BOD": "squamous_cell_carcinoma",  # Bowen's disease = SCC in situ
    "SEK": "seborrheic_keratosis",
    "MEL": "melanoma",
    "NEV": "nevus",
}

records = []

for _, row in ham_meta.iterrows():
    dx = str(row.get("dx", "")).strip().lower()
    label = HAM10000_MAP.get(dx)
    img_id = str(row.get("image_id", "")).strip()
    path = ham_images.get(img_id)
    if label and path:
        records.append({"path": path, "label": label, "source": "ham10000"})

# пронајди ја колоната со дијагноза и id во PAD-UFES-20 (имињата варираат по верзија)
diag_col = next((c for c in pad_meta.columns if c.lower() in ("diagnostic", "diagnosis", "label")), None)
id_col = next((c for c in pad_meta.columns if "img" in c.lower() and "id" in c.lower()), None)
assert diag_col and id_col, f"Провери ги колоните во PAD-UFES-20 CSV: {list(pad_meta.columns)}"

for _, row in pad_meta.iterrows():
    dx = str(row.get(diag_col, "")).strip().upper()
    label = PAD_UFES_MAP.get(dx)
    img_id = str(row.get(id_col, "")).strip()
    img_id_noext = os.path.splitext(img_id)[0]
    path = pad_images.get(img_id_noext) or pad_images.get(img_id)
    if label and path:
        records.append({"path": path, "label": label, "source": "padufes20"})

df = pd.DataFrame(records)
print("Вкупно слики по обединување:", len(df))
print(df.groupby(["label", "source"]).size().unstack(fill_value=0))

UNIFIED_CLASSES = sorted(df["label"].unique().tolist())
print("\nКатегории:", UNIFIED_CLASSES)

## Чекор 4 — Поделба train/validation и подготовка на податоците

In [ ]:
from sklearn.model_selection import train_test_split
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np

label_to_idx = {label: i for i, label in enumerate(UNIFIED_CLASSES)}
df["label_idx"] = df["label"].map(label_to_idx)

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["label_idx"], random_state=42
)
print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")

IMAGE_SIZE = 244

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        image = self.transform(image)
        return image, row["label_idx"]


train_dataset = SkinLesionDataset(train_df, train_transform)
val_dataset = SkinLesionDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# тежини по класа - HAM10000/PAD-UFES-20 се силно небалансирани (nevus доминира)
class_counts = train_df["label_idx"].value_counts().sort_index()
class_weights = (1.0 / class_counts).values
class_weights = class_weights / class_weights.sum() * len(UNIFIED_CLASSES)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Тежини по класа:", dict(zip(UNIFIED_CLASSES, class_weights.tolist())))

## Чекор 5 — Модел (EfficientNet-B3, transfer learning)

In [ ]:
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Уред:", device)

def build_model(num_classes):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(len(UNIFIED_CLASSES)).to(device)
print("Моделот е подготвен со", len(UNIFIED_CLASSES), "категории.")

## Чекор 6 — Тренирање

`EPOCHS` подолу може да го зголемиш (пр. 25-30) за подобра точност, но подолго ќе трае. 10-15 епохи се разумен почеток.

In [ ]:
import torch.optim as optim
from sklearn.metrics import f1_score
import copy, os

EPOCHS = 15
CHECKPOINT_PATH = "/content/checkpoint_best.pt"

# обиди се да монтираш Drive за checkpoint - ако успее, тренирањето
# преживува дисконекции/ресетирање на runtime (продолжува од последен најдобар модел)
DRIVE_CHECKPOINT_DIR = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/skinscan_checkpoints"
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print("Checkpoint ќе се чува и во Google Drive:", DRIVE_CHECKPOINT_DIR)
except Exception as e:
    print("Drive не е достапен - checkpoint ќе се чува само локално во Colab:", e)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = 0
best_f1 = 0.0
best_state = None

# ако веќе постои checkpoint од претходен обид, продолжи од таму наместо од почеток
resume_path = None
if DRIVE_CHECKPOINT_DIR and os.path.exists(f"{DRIVE_CHECKPOINT_DIR}/checkpoint_best.pt"):
    resume_path = f"{DRIVE_CHECKPOINT_DIR}/checkpoint_best.pt"
elif os.path.exists(CHECKPOINT_PATH):
    resume_path = CHECKPOINT_PATH

if resume_path:
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    best_f1 = ckpt.get("best_f1", 0.0)
    start_epoch = ckpt.get("epoch", 0)
    best_state = copy.deepcopy(model.state_dict())
    print(f"Продолжувам од претходен checkpoint: по епоха {start_epoch}, best_f1={best_f1:.4f}")

def save_checkpoint(epoch, state_dict, f1):
    payload = {"state_dict": state_dict, "best_f1": f1, "epoch": epoch, "class_names": UNIFIED_CLASSES}
    torch.save(payload, CHECKPOINT_PATH)
    if DRIVE_CHECKPOINT_DIR:
        try:
            torch.save(payload, f"{DRIVE_CHECKPOINT_DIR}/checkpoint_best.pt")
        except Exception as e:
            print("Не успеа зачувување во Drive:", e)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    scheduler.step()
    train_loss = running_loss / len(train_dataset)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    val_f1 = f1_score(all_labels, all_preds, average="macro")
    print(f"Епоха {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_macro_f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = copy.deepcopy(model.state_dict())
        save_checkpoint(epoch + 1, best_state, best_f1)
        print(f"  -> нов најдобар модел (f1={best_f1:.4f}) - checkpoint зачуван")

model.load_state_dict(best_state)
print(f"\nТренирањето заврши. Најдобар val macro F1: {best_f1:.4f}")

## Чекор 7 — Резултати

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=UNIFIED_CLASSES))
print("Confusion matrix (редови=точно, колони=предвидено):")
print(pd.DataFrame(confusion_matrix(all_labels, all_preds), index=UNIFIED_CLASSES, columns=UNIFIED_CLASSES))

## Чекор 8 — Зачувување и симнување на моделот

Ова создава два фајла во истиот формат каков што leafscan го користи, за директно да ги ставиш во Django проектот:

- `skin_model.pt` — тежините на моделот + список на класи
- `label_converter.json` — читливи имиња и категорија на сериозност за секоја состојба (можеш подоцна рачно да ги дотераш описите)

In [ ]:
import json, os

DISPLAY_INFO = {
    "melanoma": {"display_name": "Melanoma", "severity": "HIGH"},
    "basal_cell_carcinoma": {"display_name": "Basal Cell Carcinoma", "severity": "HIGH"},
    "squamous_cell_carcinoma": {"display_name": "Squamous Cell Carcinoma", "severity": "HIGH"},
    "actinic_keratosis": {"display_name": "Actinic Keratosis", "severity": "MEDIUM"},
    "seborrheic_keratosis": {"display_name": "Seborrheic Keratosis", "severity": "LOW"},
    "nevus": {"display_name": "Nevus (Mole)", "severity": "LOW"},
    "dermatofibroma": {"display_name": "Dermatofibroma", "severity": "LOW"},
    "vascular_lesion": {"display_name": "Vascular Lesion", "severity": "LOW"},
}

os.makedirs("/content/output", exist_ok=True)

torch.save(
    {"state_dict": model.state_dict(), "class_names": UNIFIED_CLASSES},
    "/content/output/skin_model.pt",
)

label_converter = {
    label: DISPLAY_INFO.get(label, {"display_name": label.replace("_", " ").title(), "severity": "MEDIUM"})
    for label in UNIFIED_CLASSES
}
with open("/content/output/label_converter.json", "w", encoding="utf-8") as f:
    json.dump(label_converter, f, indent=2, ensure_ascii=False)

print("Зачувано:")
print(" - /content/output/skin_model.pt")
print(" - /content/output/label_converter.json")

In [ ]:
from google.colab import files

files.download("/content/output/skin_model.pt")
files.download("/content/output/label_converter.json")

## Следен чекор — интеграција во Django

Кога ќе ги симнеш `skin_model.pt` и `label_converter.json`:

1. Стави ги во `skin_backend/analyses/services/ai_model/` (создади ја папката ако не постои) — `skin_model.pt` во таа папка, `label_converter.json` еден директориум погоре, покрај `gemini_service.py` (исто како во leafscan).
2. Користи го `skin_model_service.py` (го добиваш одделно) наместо `ai_model_service.py` — веќе е прилагоден да враќа едно име на состојба + confidence, наместо plant/disease пар.
3. Gemini делот (`gemini_service.py`) останува речиси идентичен на leafscan — само промени го промптот да генерира објаснување + општи совети + препорака за лекар, наместо третмани за растение.

Ако точноста на моделот не те задоволува:
- Зголеми `EPOCHS` во Чекор 6
- Пробај да собереш повеќе слики (особено за поретките категории како dermatofibroma/vascular_lesion)
- Разгледај да додадеш и DDI или Fitzpatrick17k датасет за подобра распределеност на тонови на кожа

Секогаш тестирај со реални телефонски фотографии пред да веруваш во точноста — HAM10000 е дерматоскопски датасет и моделот сепак може да работи послабо на обични фотки отколку на dermoscope слики.